## Cluster Goals from Repeated Sampling

This notebook clusters the goals acquired from repeated sampling of the goal extraction task. Clustering allows for the consolidation of duplicate goals into the same cluster, which can then be sampled to reduce redundancy.

In [1]:
#!pip install sentence_transformers scikit-learn

In [2]:
import json
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
import numpy as np

data_path = 'data1_gpt52'

goals = json.load(open(f'{data_path}/extracted-goals.json'))

C:\Python313\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
for i in goals.keys():
    unique = set()
    for j in range(len(goals[i])):
        unique.update(goals[i][j])
    print('Unique goals in %s: %i' % (i, len(unique)))

Unique goals in 0: 76
Unique goals in 1: 104
Unique goals in 2: 79
Unique goals in 3: 159
Unique goals in 4: 98
Unique goals in 5: 65
Unique goals in 6: 158
Unique goals in 7: 75
Unique goals in 8: 162
Unique goals in 9: 104
Unique goals in 10: 73
Unique goals in 11: 178
Unique goals in 12: 48
Unique goals in 13: 139
Unique goals in 14: 144
Unique goals in 15: 79
Unique goals in 16: 131
Unique goals in 17: 78
Unique goals in 18: 52
Unique goals in 19: 180
Unique goals in 20: 106
Unique goals in 21: 50
Unique goals in 22: 123
Unique goals in 23: 58
Unique goals in 24: 42
Unique goals in 25: 112
Unique goals in 26: 159
Unique goals in 27: 342
Unique goals in 28: 163
Unique goals in 29: 119
Unique goals in 30: 105
Unique goals in 31: 117
Unique goals in 32: 260
Unique goals in 33: 90


## Define clustering functions

In [4]:


embedder = SentenceTransformer('paraphrase-MiniLM-L6-v2')

def cluster(corpus):
    # normalize the embeddings to unit vectors
    embeddings = embedder.encode(corpus)
    embeddings = embeddings /  np.linalg.norm(embeddings, axis=1, keepdims=True)
    
    # cluster
    model = AgglomerativeClustering(n_clusters=None, distance_threshold=1.5)
    model.fit(embeddings)
    
    clustered = {}
    for sentence_id, cluster_id in enumerate(model.labels_):
        if cluster_id not in clustered:
            clustered[cluster_id] = []
        clustered[cluster_id].append(corpus[sentence_id])
    return clustered

def print_cluster(clustered):   
    for i, cluster in clustered.items():
        print("Cluster ", i + 1)
        print(cluster)
        print("")

def find_cluster(item, clustered):
    for i, c in clustered.items():
        if item in c:
            return i, c
    return -1, None

def compute_change(cluster0, cluster1):
    set0 = set(cluster0)
    set1 = set(cluster1)
    gain = len(set1 - set0) / len(set1)
    loss = len(set0 - set1) / len(set0)
    return gain, loss

In [5]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def compare(g1, g2):
    e1 = model.encode(g1, convert_to_tensor=True)
    e2 = model.encode(g2, convert_to_tensor=True)
    t = util.pytorch_cos_sim(e1, e2)
    return float(t.mean())

def is_similar(g1, g_list, min_distance=0.7):
    similar = []
    for g2 in g_list:
        if compare(g1, g2) >= min_distance:
            similar.append(g2)
    return similar

def sub_cluster(cluster, min_distance=0.7):
    subs = [list([cluster[0]])]
    for i in range(1, len(cluster)):
        p = cluster[i]
        matched = False
        for j in range(len(subs)):
            for q in subs[j]:
                if compare(p, q) >= min_distance:
                    if not p in subs[j]:
                        subs[j].append(p)
                    matched = True
                    break
            if matched:
                break
        if not matched:
            subs.append([p])
    return subs

In [6]:
import random

sampled = {}
for i in goals.keys():
    print('Generating cluster for goals %s' % i)

    corpus = []
    for j in range(len(goals[i])):
        corpus.extend(goals[i][j])

    if len(corpus) == 0:
        continue

    sampled[i] = []
    clusters = cluster(corpus)
    clusters = {int(k):v for k,v in clusters.items()}
    json.dump(clusters, open('%s/cache/clusters-%s.json' % (data_path, i), 'w+'))
    
    for j, c in clusters.items():
        g = random.sample(c, 1)
        sampled[i].append(g[0])

Generating cluster for goals 0
Generating cluster for goals 1
Generating cluster for goals 2
Generating cluster for goals 3
Generating cluster for goals 4
Generating cluster for goals 5
Generating cluster for goals 6
Generating cluster for goals 7
Generating cluster for goals 8
Generating cluster for goals 9
Generating cluster for goals 10
Generating cluster for goals 11
Generating cluster for goals 12
Generating cluster for goals 13
Generating cluster for goals 14
Generating cluster for goals 15
Generating cluster for goals 16
Generating cluster for goals 17
Generating cluster for goals 18
Generating cluster for goals 19
Generating cluster for goals 20
Generating cluster for goals 21
Generating cluster for goals 22
Generating cluster for goals 23
Generating cluster for goals 24
Generating cluster for goals 25
Generating cluster for goals 26
Generating cluster for goals 27
Generating cluster for goals 28
Generating cluster for goals 29
Generating cluster for goals 30
Generating cluster

In [7]:
json.dump(sampled, open('%s/sampled-goals.json' % data_path, 'w'))

In [8]:
clusters = json.load(open('%s/cache/clusters-4.json' % data_path, 'r'))
for i in sorted(clusters.keys()):
    print('%s: %s' % (i, clusters[i]))

0: ['Return later to complete tasks associated with tagged emails', 'Return later to complete follow-up tasks', 'Work on deadlines', 'Send bulk messages to a large number of people within a short span of time', 'Schedule emails to be sent at different times', 'Schedule messages to be sent at different times']
1: ['Help others get a deeper view of how users want to use an email tool', 'Open a web browser to access email', 'Open a web browser to access email', 'Use an assistant to help read emails', 'Use an assistant to help write emails', 'Use an email client mostly for work', 'Use an email client mostly for work']
10: ['Receive message excerpts rather than miss important messages', 'Notice important instructions or disclaimers in messages before taking action', 'Avoid wasting time by missing important information in messages', 'Receive fewer messages that summarize the content of many messages', 'Receive notifications for important messages', 'Be able to identify important messages amo

### Prompt-based Summarization

The following cells study the use of prompts to reduce redundancy in goals.

In [9]:
from openai import OpenAI

client = OpenAI()

def prompt_model(prompt):
    response = client.chat.completions.create(
      model="gpt-5.2-2025-12-11",
      #model="gpt-4o-2024-08-06",
      #model='o3-mini-2025-01-31',
      #model='gpt-4o-mini',
      messages=[
        {
          "role": "system",
          "content": "You are a helpful assistant."
        },
        {
          "role": "user",
          "content": prompt
        }
      ],
      response_format= {"type": "json_object"}
    )
    return response.choices[0].message.content

In [16]:
prompt = """Read the following list of goals and summarize the list into as few goal statements as possible by removing any duplicate or similar goals. When summarizing a goal, do not reference any specific product or service name, instead describe the technology provided by that product or service. Each goal should describe one action. Avoid describing the development of software and focus only on what software should do. Respond with the goals in a JSON list. Do not comment or elaborate.

List: %s

Summaries: """

def get_json(r):
    i = r.find('```json')
    j = r.find('```', i + 1)
    if i >= 0 and j > i:
        return r[i+7:j]
    else:
        return r  # in gpt 5.2, no json blocks since response format can be specified

summarized = {}
for i in goals.keys():
    print('Generating cluster for goals %s' % i)

    corpus = []
    for j in range(len(goals[i])):
        corpus.extend(goals[i][j])

    if len(corpus) == 0:
        continue

    clusters = cluster(corpus)
    
    summarized[i] = []
    for j, c in clusters.items():
        p = prompt % c
        r = prompt_model(p)
        #print(p)
        #print(r)
        #print()
        g = json.loads(get_json(r))
        summarized[i].extend(g)

Generating cluster for goals 0
Generating cluster for goals 1
Generating cluster for goals 2
Generating cluster for goals 3
Generating cluster for goals 4
Generating cluster for goals 5
Generating cluster for goals 6
Generating cluster for goals 7
Generating cluster for goals 8
Generating cluster for goals 9
Generating cluster for goals 10
Generating cluster for goals 11
Generating cluster for goals 12
Generating cluster for goals 13
Generating cluster for goals 14
Generating cluster for goals 15
Generating cluster for goals 16
Generating cluster for goals 17
Generating cluster for goals 18
Generating cluster for goals 19
Generating cluster for goals 20
Generating cluster for goals 21
Generating cluster for goals 22
Generating cluster for goals 23
Generating cluster for goals 24
Generating cluster for goals 25
Generating cluster for goals 26
Generating cluster for goals 27
Generating cluster for goals 28
Generating cluster for goals 29
Generating cluster for goals 30
Generating cluster

In [17]:
for i in summarized.keys():
    unique = set(summarized[i])
    print('Unique goals in %s: %i' % (i, len(unique)))

Unique goals in 0: 37
Unique goals in 1: 47
Unique goals in 2: 42
Unique goals in 3: 79
Unique goals in 4: 56
Unique goals in 5: 31
Unique goals in 6: 78
Unique goals in 7: 37
Unique goals in 8: 78
Unique goals in 9: 51
Unique goals in 10: 42
Unique goals in 11: 95
Unique goals in 12: 30
Unique goals in 13: 63
Unique goals in 14: 69
Unique goals in 15: 29
Unique goals in 16: 63
Unique goals in 17: 37
Unique goals in 18: 30
Unique goals in 19: 84
Unique goals in 20: 58
Unique goals in 21: 25
Unique goals in 22: 61
Unique goals in 23: 23
Unique goals in 24: 18
Unique goals in 25: 59
Unique goals in 26: 83
Unique goals in 27: 154
Unique goals in 28: 88
Unique goals in 29: 65
Unique goals in 30: 61
Unique goals in 31: 64
Unique goals in 32: 124
Unique goals in 33: 45


In [18]:
json.dump(summarized, open('%s/summarized.json' % data_path, 'w+'))

In [19]:
for g in summarized['14']:
    print(g)

Receive personalized, relevant, and timely notifications about job opportunities based on preferred domains and skills
Access an up-to-date list of relevant job opportunities
Save or record job postings and their identifiers for later review or application
Retrieve the application link for a job posting after indicating interest
View feedback from current employees about employers or roles
Receive feedback on the outcome or status after submitting a job application
Receive job notification emails that are concise and include all necessary job details
Apply to a job directly from a job notification email
Receive relevant job notifications on a mobile device without being overwhelmed
Prevent unsolicited job notifications
Prevent unsolicited job-related contact from unknown people
Minimize the time spent navigating between different online portals to apply for jobs
Reduce delays in retrieving the application link from a job posting
Prevent applying to job openings that are no longer curre